# PDF RAG System

This notebook builds a Retrieval-Augmented Generation (RAG) system that can answer questions from a PDF file.

We will:
- Load PDF
- Extract text
- Chunk text
- Create embeddings
- Store in FAISS
- Retrieve relevant chunks
- Generate answers

In [8]:
import fitz
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [9]:
pdf_path = "Datasets/sample.pdf"  

doc = fitz.open(pdf_path)

text = ""

for page in doc:
    text += page.get_text()

print(text[:1000])

 
  
Abstract—This study aims to use computers to detect and 
recognize ventilation objects (masks and tubes) and their 
positions on the patient’s face. We created two models: the You 
Only Look Once (YOLO) and the Transfer Learning (TL) 
models, to perform this computer vision task. The development 
processes and comparison of performance will be described in 
this paper. The TL model had a better performance (98%) 
compared to the YOLO model (93%).  
 
Clinical Relevance— Healthcare providers and researchers 
interested in the field of computer vision applied in medicine, 
specifically automatic object detection using video streams or 
real-time video streaming may benefit from findings reported. 
I. INTRODUCTION 
Digital Video Recordings (DVRs) have played an 
important role in patient care, from preventive to intensive [1-
3].  For example, analyzing recordings of gait helped identify 
unsafe habits and reduce incidence of falls [4]. Furthermore, 
DVRs have been implemented in hea

We split PDF text into smaller chunks for better retrieval.

In [ ]:
def chunk_text(text, chunk_size=500, overlap=100):

    chunks = []
    step = chunk_size - overlap

    for i in range(0, len(text), step):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)

    return chunks

In [13]:
chunks = chunk_text(text)

print('Total chunks:', len(chunks))
print(chunks[0])

Total chunks: 60
 
  
Abstract—This study aims to use computers to detect and 
recognize ventilation objects (masks and tubes) and their 
positions on the patient’s face. We created two models: the You 
Only Look Once (YOLO) and the Transfer Learning (TL) 
models, to perform this computer vision task. The development 
processes and comparison of performance will be described in 
this paper. The TL model had a better performance (98%) 
compared to the YOLO model (93%).  
 
Clinical Relevance— Healthcare providers


In [14]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [16]:
chunks_embeddings = model.encode(chunks)

chunk_embedding = np.array(chunks_embeddings).astype('float32')

print(chunk_embedding.shape)

(60, 384)


In [17]:
dimension = chunk_embedding.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(chunk_embedding)

print('vector in DB: ', index.ntotal)

vector in DB:  60


In [18]:
chunk_embedding.shape

(60, 384)

In [ ]:
def retrieve(query, k=3):

    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding).astype('float32')

    distance,indices = index.search(query_embedding, k)

    results = [chunks[i] for i in indices[0]]

    return results  

In [20]:
def build_context(retrieved_chunks):

    return "\n\n".join(retrieved_chunks)

In [21]:
def fake_llm(prompt):

    return "Answer generated from PDF context (replace with real LLM later)."

In [22]:
def create_prompt(context, query):

    return f"""
Context:
{context}

Question:
{query}

Answer:
"""

In [23]:
def pdf_rag(query):

    retrieved = retrieve(query, k=3)

    context = build_context(retrieved)

    prompt = create_prompt(context, query)

    answer = fake_llm(prompt)

    return {
        "query": query,
        "retrieved_chunks": retrieved,
        "context": context,
        "answer": answer
    }

In [24]:
pdf_rag("What is this document about?")


{'query': 'What is this document about?',
 'retrieved_chunks': ['n a patient was incubated, the recruiters had to \nseek permission from the patient’s family. A consent form had \nto be signed by a family member if she/he permitted the \nrecordings and agreed to participate in the study. After the \npatient was extubated, the recruiters then had to explain the \nstudy and the recording process to the patients. If the patient \nagreed to participate, she/he had to sign a consent form.  \nPatients’ participants are voluntary, and patients have the right \nto withdraw h',
  'piratory support \nC.1.2 \nObject Code \nat T+1 > \nObject Code \nat T \n   a. No respiratory support needed. b. Ordered by the advancement in need of respiratory support (advancement levels). \nB.  Study Subjects \nPatients who were admitted to the ICU and volunteered for \nthis study. Patients or their families had to sign the consent \nform and have a clear understanding about the recording \nprocess. When a patien

In [25]:
pdf_rag("Explain the main topic")

{'query': 'Explain the main topic',
 'retrieved_chunks': [' deterioration \nand timely intervention to prevent poor outcomes continues to \nbe the focus of research. \n \n*Research approved by Mayo Clinic institutional review board. \nJ. Chaudri is with the Department of Computer Science, Marshall \nUniversity, Huntington, WV, 55902, USA. (phone: 304-696-2694; e-mail: \nchaudri@marshall.edu).  \nII. BACKGROUND \nImage classification is the process of assigning a specific \nclass (also called label or class-label) to an image (or a video \nframe). Object localizati',
  ' an \neffective AI model since it decides the generalization ability of \nan AI model. For this project, group discussions helped us \nbrainstorming a clear recording plan in advance in order to be \nable to collect images at different angles, backgrounds, \nlighting conditions and noises.  \na. \nCamera Types \nThere were 2 types of cameras which were used for \nrecording the videos. The focus camera was used to collect